# Solving System of Linear Equations
The gradient Descent mechanism is not just stable for Linear regression, or Machine Learning - but also to find solution to any system of equations (or constraints), using the error-based learning approach.

The process starts with an initial guess, and then iterates on. BackPropagation - based on the Loss calculation, the updates (gradients and the deltas) are driven backwards towards selecting a new and a better guess on each iteration, eventually converging to the solution (x0, y0).


## OPTION-A: Using Torch (tensors)

In [8]:
# Solving the system of linear eqn in 2 variables (x, y)
# 2x + 3y = 13
#  x + 2y = 8

import numpy as np
import torch
from torch import tensor

# Initial guess
x,y = 1,1

A_2x2 = np.matrix([
    [2, 3],
    [1, 2],
])

X_2x1 = np.matrix([x,y]).reshape((2,1))
B_2x1 = np.matrix([13,8]).reshape((2,1))

# Objective
print("Objective: to solve")
print(f"A @ X = B\n")


Objective: to solve
A @ X = B



In [26]:
# Dependency graph:
# X ==> Loss

# Only Tensors of floating point and complex dtype can require gradients
t_A = tensor(A_2x2, requires_grad=False, device="cpu", dtype=torch.float32)
t_B = tensor(B_2x1, requires_grad=False, device="cpu", dtype=torch.float32)
t_X = tensor(X_2x1, requires_grad=True, device="cpu", dtype=torch.float32)
# print(f"A = {t_A}\n")
# print(f"B = {t_B}\n")

# Loss (initial)
# Loss = squared mean of A @ X - B
loss = torch.mean(torch.pow((t_A @ t_X) - t_B, 2))
print(loss)
print(t_X)
print(t_A)
print(t_B)

tensor(44.5000, grad_fn=<MeanBackward0>)
tensor([[1.],
        [1.]], requires_grad=True)
tensor([[2., 3.],
        [1., 2.]])
tensor([[13.],
        [ 8.]])


In [35]:
epochs = 100
learning_rate = 0.01
for epoch in range(epochs):
    # Find the current loss
    print(f"Guess Number: #{epoch+1}/{epochs} (x,y)=\n{t_X.data}")
    loss = torch.mean(torch.pow((t_A @ t_X) - t_B, 2))
    print(f"Loss (Error) = {loss}\n\n")

    # Back propagate the loss(error)
    loss.backward()

    # Update the X (without the grad setting)
    with torch.no_grad():
        t_X -= learning_rate * t_X.grad

    # Reset grads
    t_X.grad.zero_()

    # GRAVE MISTAKE: Make the Grad = 0, not the Matrix itself.
    # "RuntimeError: a leaf Variable that requires grad is being used in an in-place operation."
    # t_X.zero_()

    # Grave MISTAKE - don't make the grad Constant Matrices = 0!
    # t_A.grad.zero_()
    # t_B.grad.zero_()

print(f"x,y = {torch.round(t_X)}\n")

Guess Number: #1/100 (x,y)=
tensor([[1.9289],
        [3.0440]])
Loss (Error) = 0.00019484086078591645


Guess Number: #2/100 (x,y)=
tensor([[1.9289],
        [3.0439]])
Loss (Error) = 0.00019462138880044222


Guess Number: #3/100 (x,y)=
tensor([[1.9289],
        [3.0439]])
Loss (Error) = 0.00019440204778220505


Guess Number: #4/100 (x,y)=
tensor([[1.9290],
        [3.0439]])
Loss (Error) = 0.00019418283773120493


Guess Number: #5/100 (x,y)=
tensor([[1.9290],
        [3.0439]])
Loss (Error) = 0.00019396374409552664


Guess Number: #6/100 (x,y)=
tensor([[1.9291],
        [3.0438]])
Loss (Error) = 0.00019376075943000615


Guess Number: #7/100 (x,y)=
tensor([[1.9291],
        [3.0438]])
Loss (Error) = 0.00019354188407305628


Guess Number: #8/100 (x,y)=
tensor([[1.9291],
        [3.0438]])
Loss (Error) = 0.00019332923693582416


Guess Number: #9/100 (x,y)=
tensor([[1.9292],
        [3.0438]])
Loss (Error) = 0.00019312048971187323


Guess Number: #10/100 (x,y)=
tensor([[1.9292],
        

## OPTION B: Using NN library

### Demystifying `torch.nn.Linear`: Under the Hood

When building Neural Networks, the `nn.Linear` layer is the workhorse that performs the affine transformation. While standard textbooks define this as $y = Wx + b$, PyTorch processes data in **batches**, optimizing memory by using the following equation:

$$y = x A^T + b$$

---

#### 1. The Matrix Dimensions (The Anatomy)

Assume we define a layer as `layer = nn.Linear(in_features=3, out_features=2)` and pass in a batch of 4 samples.

* **The Input ($x$):** Shape is `[batch_size, in_features]`.
  * *Example:* `[4, 3]`. You have 4 rows of data, and each row has 3 features.
* **The Weight Matrix ($A$ or $W$):** Shape is `[out_features, in_features]`.
  * *Example:* `[2, 3]`. PyTorch stores the weights in this "wide" format. It has 2 rows (one for each output neuron) and 3 columns (one for each input feature).
* **The Bias Vector ($b$):** Shape is `[out_features]`.
  * *Example:* `[2]`. A 1D tensor with 2 numbers.

##### The 3-Step Calculation
1. **The Transpose:** PyTorch transposes its internal weight matrix from `[2, 3]` to `[3, 2]`.
2. **Matrix Multiplication:** It calculates the dot product of the input `[4, 3]` and the transposed weights `[3, 2]`, resulting in a `[4, 2]` matrix.
3. **Broadcasting:** It stretches the `[2]` bias vector across all 4 rows and adds it to the result.

---

#### 2. Where Do the Initial Numbers Come From?

When you instantiate the layer, PyTorch automatically fills the $W$ and $b$ matrices using a **Uniform Distribution** bounded by $\pm \frac{1}{\sqrt{\text{in\_features}}}$.

For `in_features = 3`, it picks random numbers between $-\frac{1}{\sqrt{3}}$ and $+\frac{1}{\sqrt{3}}$ (roughly -0.577 to +0.577). This prevents "exploding" or "vanishing" gradients on the very first day of training.

---

#### 3. Controlling Weights and Biases

Weights and biases are stored as `torch.nn.Parameter` objects. You have full control over how they are initialized.

### Option A: Built-in Initializers (The ML Way)
Use `torch.nn.init` to apply industry-standard techniques like Xavier/Glorot or Kaiming/He.

```python
import torch
import torch.nn as nn

layer = nn.Linear(3, 2)

### Overwrite weights with Xavier Normal (Great for Tanh/Sig
```
### Option B: Injecting Custom Matrices (For Debugging/Teaching)
You can force the layer to use exact numbers by modifying the .data directly inside a torch.no_grad() block.
```python
import torch
import torch.nn as nn

layer = nn.Linear(3, 2)

# Create custom matrices (Must match the expected shapes!)
custom_weights = torch.tensor([[1.0, 2.0, 3.0],
                               [4.0, 5.0, 6.0]])
custom_bias = torch.tensor([10.0, 20.0])

# Inject them directly
with torch.no_grad():
    layer.weight.copy_(custom_weights)
    layer.bias.copy_(custom_bias)
```
### Option C: Disabling the Bias Completely
If you are placing a Linear layer immediately before a Normalization layer (like BatchNorm), the bias is mathematically redundant. You can disable it entirely.
```python
# Create a layer with ONLY a weight matrix, no bias.
layer_no_bias = nn.Linear(in_features=3, out_features=2, bias=False)

print(layer_no_bias.bias) # Output: None
```

In [5]:
 # Just a warm-up
import torch
import torch.nn as nn

# We have input as 3 dims, and output has 2 dims.
linear_layer = nn.Linear(3, 2)
print(linear_layer)
# 10 data points - each data-point ==> (x1, x2, x3)
input = torch.randn(10, 3)
print(f"Input 10x3: \n{input}")

# It does this: Y_hat = X.W^T + b
# Where Y_hat is the output matrix, and X is the input matrix
output = linear_layer(input)

print(f"Output 10X2: \n{output}")
print(output.size())

print()
# Now let's Inspect the internal matrices PyTorch created
print("Weight shape (W):", linear_layer.weight.shape)
print(f"Weight (W): {linear_layer.weight}")
print()
print("Bias shape (b):", linear_layer.bias.shape)
print(f"Bias (b): {linear_layer.bias}")
# Output: torch.Size([2])

# Let's now understand the parameters
print()
print(f"Parameters: {linear_layer._parameters}\n")


Linear(in_features=3, out_features=2, bias=True)
Input 10x3: 
tensor([[ 2.0940,  1.1740,  1.0054],
        [-0.1970, -0.5678,  0.1384],
        [-1.7774,  1.4650, -0.0672],
        [-0.2573, -1.5342,  0.3070],
        [-0.1165, -1.7325,  0.4073],
        [-1.1675,  1.0353,  1.0544],
        [-0.2434, -0.8994,  0.3575],
        [-0.3476,  0.2629,  0.4767],
        [-1.0270,  0.6631,  0.5538],
        [-0.9480, -0.2999, -0.1769]])
Output 10X2: 
tensor([[ 0.4145, -0.5285],
        [-0.4023, -0.2371],
        [-0.9614, -0.0351],
        [-0.4555, -0.3123],
        [-0.4142, -0.3531],
        [-0.8459, -0.3852],
        [-0.4467, -0.3055],
        [-0.4830, -0.2941],
        [-0.7465, -0.2682],
        [-0.6555, -0.1051]], grad_fn=<AddmmBackward0>)
torch.Size([10, 2])

Weight shape (W): torch.Size([2, 3])
Weight (W): Parameter containing:
tensor([[ 0.3844,  0.0136, -0.1009],
        [-0.0494,  0.0333, -0.2725]], requires_grad=True)

Bias shape (b): torch.Size([2])
Bias (b): Parameter contai

# Solving Linear Equations with PyTorch

At its core, a Neural Network is a mathematical engine that solves massively complex systems of equations using gradient descent. By using PyTorch to solve a simple $2 \times 2$ linear system, we can observe the exact training loop that powers large language models, just on a microscopic scale.

## The Intuition: Mapping Math to PyTorch

To solve a system of equations using `nn.Linear`, we map the algebraic components into the PyTorch affine transformation formula: $\text{Output} = \text{Input} \times W^T + b$.

Given the system:
1. $2x + 3y = 13$
2. $1x + 2y = 8$

### The Mapping:
* **The Input (Features):** The coefficients of the equations become the input dataset (a batch of 2 samples, where each sample has 2 features).
  $$A = \begin{bmatrix} 2 & 3 \\ 1 & 2 \end{bmatrix}$$
* **The Weights ($W$):** The unknown variables $x$ and $y$ become the learnable parameters (weights) of the `nn.Linear` layer.
  $$W = \begin{bmatrix} x & y \end{bmatrix}$$
* **The Bias ($b$):** The equations do not have a standalone constant added to the left side, so we set `bias=False`.
* **The Target Output ($B$):** The right side of the equations becomes the "Ground Truth" we want the network to predict.
  $$B = \begin{bmatrix} 13 \\ 8 \end{bmatrix}$$

---

## The PyTorch Implementation

This script defines the `nn.Module`, sets up the Adam optimizer, and loops until it finds the exact solution through gradient descent.

```python
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Define the Dataset
# Input A: Shape [2, 2] -> 2 equations, 2 coefficients each
A = torch.tensor([[2.0, 3.0],
                  [1.0, 2.0]])

# Target B: Shape [2, 1] -> 2 target answers
B = torch.tensor([[13.0],
                  [8.0]])

# 2. Define the Neural Network Module
class EquationSolver(nn.Module):
    def __init__(self):
        super(EquationSolver, self).__init__()
        # in_features=2 (the coefficients)
        # out_features=1 (the single result of the equation)
        # bias=False because our equations don't have a standalone + C term
        self.layer = nn.Linear(in_features=2, out_features=1, bias=False)

    def forward(self, inputs):
        return self.layer(inputs)

# 3. Instantiate the model, loss function, and optimizer
model = EquationSolver()

# Mean Squared Error: Computes how far our (2x+3y) prediction is from 13
criterion = nn.MSELoss()

# Adam Optimizer: Updates the [x, y] weights to minimize the loss
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 4. The Training Loop
epochs = 500

print("--- Starting Training ---")
for epoch in range(epochs):
    # Step A: Forward Pass (Calculate current predictions)
    predictions = model(A)

    # Step B: Calculate the Loss (MSE between predictions and target B)
    loss = criterion(predictions, B)

    # Step C: Backward Pass & Optimization
    optimizer.zero_grad() # Clear old gradients
    loss.backward()       # Calculate new gradients via backpropagation
    optimizer.step()      # Update the weights (x, y)

    # Print progress
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

# 5. Extract the Final Answer
print("\n--- Final Results ---")
# The weights are stored in a [1, 2] matrix. We detach and flatten it.
final_weights = model.layer.weight.detach().flatten()

x_solved = final_weights[0].item()
y_solved = final_weights[1].item()

print(f"Calculated x : {x_solved:.4f} (Expected: 2.0)")
print(f"Calculated y : {y_solved:.4f} (Expected: 3.0)")
```

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. Define the Dataset
# Input A: Shape [2, 2] -> 2 equations, 2 coefficients each
A = torch.tensor([[2.0, 3.0],
                  [1.0, 2.0]])

# Target B: Shape [2, 1] -> 2 target answers
B = torch.tensor([[13.0],
                  [8.0]])

# 2. Define the Neural Network Module
class EquationSolver(nn.Module):
    def __init__(self):
        super(EquationSolver, self).__init__()
        # in_features=2 (the coefficients)
        # out_features=1 (the single result of the equation)
        # bias=False because our equations don't have a standalone + C term
        self.layer = nn.Linear(in_features=2, out_features=1, bias=False)

    def forward(self, inputs):
        return self.layer(inputs)

# 3. Instantiate the model, loss function, and optimizer
model = EquationSolver()

# Mean Squared Error: Computes how far our (2x+3y) prediction is from 13
criterion = nn.MSELoss()

# Adam Optimizer: Updates the [x, y] weights to minimize the loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model.layer._parameters)

# 4. The Training Loop
epochs = 50000

print("--- Starting Training ---")
for epoch in range(epochs):
    # Step A: Forward Pass (Calculate current predictions)
    predictions = model(A)

    # Step B: Calculate the Loss (MSE between predictions and target B)
    loss = criterion(predictions, B)

    # Step C: Backward Pass & Optimization
    optimizer.zero_grad() # Clear old gradients
    loss.backward()       # Calculate new gradients via backpropagation
    optimizer.step()      # Update the weights (x, y)

    # Print progress
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

# 5. Extract the Final Answer
print("\n--- Final Results ---")
# The weights are stored in a [1, 2] matrix. We detach and flatten it.
final_weights = model.layer.weight.detach().flatten()

x_solved = final_weights[0].item()
y_solved = final_weights[1].item()

print(f"Calculated x : {x_solved:.4f} (Expected: 2.0)")
print(f"Calculated y : {y_solved:.4f} (Expected: 3.0)")

{'weight': Parameter containing:
tensor([[ 0.5378, -0.6376]], requires_grad=True), 'bias': None}
--- Starting Training ---
Epoch [100/50000], Loss: 124.684669
Epoch [200/50000], Loss: 115.853470
Epoch [300/50000], Loss: 107.495232
Epoch [400/50000], Loss: 99.591171
Epoch [500/50000], Loss: 92.123276
Epoch [600/50000], Loss: 85.074249
Epoch [700/50000], Loss: 78.427483
Epoch [800/50000], Loss: 72.167007
Epoch [900/50000], Loss: 66.277443
Epoch [1000/50000], Loss: 60.743965
Epoch [1100/50000], Loss: 55.552258
Epoch [1200/50000], Loss: 50.688499
Epoch [1300/50000], Loss: 46.139290
Epoch [1400/50000], Loss: 41.891624
Epoch [1500/50000], Loss: 37.932877
Epoch [1600/50000], Loss: 34.250751
Epoch [1700/50000], Loss: 30.833267
Epoch [1800/50000], Loss: 27.668701
Epoch [1900/50000], Loss: 24.745594
Epoch [2000/50000], Loss: 22.052708
Epoch [2100/50000], Loss: 19.579006
Epoch [2200/50000], Loss: 17.313614
Epoch [2300/50000], Loss: 15.245886
Epoch [2400/50000], Loss: 13.365263
Epoch [2500/50000],